# StageBridge V1: GRANULAR Visual Pipeline

**Watch every step with figures!**

This notebook runs the full pipeline with visualizations at EVERY step:
- Synthetic data generation with ground truth visualization
- Stage distributions and donor structure
- Niche influence ground truth
- Training progress with loss curves (live)
- Latent space projections per epoch
- Ground truth recovery metrics

In [ ]:
# ============================================================================
# CELL 1: SETUP AND CONFIGURATION
# ============================================================================
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import torch
import warnings
from IPython.display import display, clear_output
from datetime import datetime

# Path setup
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

# Configuration
DIFFICULTY = "hard"  # easy/medium/hard
N_CELLS = 2000
N_DONORS = 10
LATENT_DIM = 32
N_EPOCHS = 20
N_FOLDS = 3
BATCH_SIZE = 32
SEED = 42

# Paths
OUTPUT_DIR = Path(f"outputs/granular_{DIFFICULTY}_{datetime.now().strftime('%Y%m%d_%H%M')}")
DATA_DIR = OUTPUT_DIR / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("STAGEBRIDGE V1 GRANULAR PIPELINE")
print("=" * 80)
print(f"Difficulty: {DIFFICULTY.upper()}")
print(f"  - easy: Low noise, strong signals")
print(f"  - medium: Balanced")
print(f"  - hard: High noise, challenging recovery")
print(f"Cells: {N_CELLS}, Donors: {N_DONORS}")
print(f"Training: {N_EPOCHS} epochs x {N_FOLDS} folds")
print(f"Output: {OUTPUT_DIR}")
print("=" * 80)

---
## STEP 1: Generate Synthetic Data
Generate synthetic data with known ground truth for all 4 AGENTS.md suites

In [ ]:
# ============================================================================
# CELL 2: GENERATE SYNTHETIC DATA
# ============================================================================
print("\n" + "="*80)
print("STEP 1: GENERATING SYNTHETIC DATA")
print("="*80)

from stagebridge.data.synthetic import generate_synthetic_v2, SyntheticConfig

print(f"\nGenerating {N_CELLS} cells with {DIFFICULTY} difficulty...")
print("This includes ground truth for:")
print("  - Suite A: Flow field dynamics (stage transitions)")
print("  - Suite B: Niche influence vectors (sender->receiver effects)")
print("  - Suite C: Clone structure (evolutionary compatibility)")
print("  - Suite D: Spatial interaction rules\n")

data_path = generate_synthetic_v2(
    output_dir=str(DATA_DIR),
    n_cells=N_CELLS,
    n_donors=N_DONORS,
    latent_dim=LATENT_DIM,
    difficulty=DIFFICULTY,
    seed=SEED,
)

print(f"\nData generated at: {data_path}")

# List generated files
print("\nGenerated files:")
for f in sorted(data_path.glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name}: {size:.1f} KB")

In [ ]:
# ============================================================================
# CELL 3: LOAD AND INSPECT DATA
# ============================================================================
print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

# Load all tables
cells_df = pd.read_parquet(data_path / "cells.parquet")
neighborhoods_df = pd.read_parquet(data_path / "neighborhoods.parquet")
transitions_df = pd.read_parquet(data_path / "transitions.parquet")
stage_edges_df = pd.read_parquet(data_path / "stage_edges.parquet")

with open(data_path / "ground_truth.json") as f:
    ground_truth = json.load(f)

with open(data_path / "split_manifest.json") as f:
    splits = json.load(f)

print(f"\nCells: {len(cells_df):,}")
print(f"Neighborhoods: {len(neighborhoods_df):,}")
print(f"Transitions: {len(transitions_df):,}")
print(f"Stage edges: {len(stage_edges_df)}")
print(f"\nDonors: {cells_df['donor_id'].nunique()}")
print(f"Stages: {cells_df['stage'].nunique()}")
print(f"Cell types: {cells_df['cell_type'].nunique()}")
print(f"Clones: {cells_df['clone_id'].nunique()}")

print("\nGround truth keys:")
for key in ground_truth.keys():
    print(f"  - {key}")

---
## STEP 2: Visualize Data Distribution

In [ ]:
# ============================================================================
# CELL 4: FIGURE - STAGE DISTRIBUTION
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Stage Distribution")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Stage counts
stage_counts = cells_df['stage'].value_counts().sort_index()
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(stage_counts)))
stage_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title("Cells per Stage", fontweight='bold', fontsize=12)
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(stage_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=9)

# Donors per stage
donors_per_stage = cells_df.groupby('stage')['donor_id'].nunique().sort_index()
donors_per_stage.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title("Donors per Stage", fontweight='bold', fontsize=12)
axes[1].set_ylabel("Count")
axes[1].tick_params(axis='x', rotation=45)

# Stage graph (progression)
stages = ground_truth.get('stages', list(stage_counts.index))
stage_edges = ground_truth.get('stage_edges', [])

# Draw stage progression
ax = axes[2]
y_pos = {stage: i for i, stage in enumerate(stages)}
for stage in stages:
    ax.scatter([0], [y_pos[stage]], s=300, c=[colors[y_pos[stage]]], edgecolor='black', zorder=3)
    ax.text(0.15, y_pos[stage], stage, va='center', fontsize=11)

for src, tgt in stage_edges:
    if src in y_pos and tgt in y_pos:
        ax.annotate('', xy=(0, y_pos[tgt]), xytext=(0, y_pos[src]),
                   arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_xlim(-0.5, 1)
ax.set_ylim(-0.5, len(stages) - 0.5)
ax.set_title("Stage Progression Graph", fontweight='bold', fontsize=12)
ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig1_stage_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELL 5: FIGURE - DONOR STRUCTURE
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Donor and Clone Structure")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Cells per donor
donor_counts = cells_df['donor_id'].value_counts().sort_index()
donor_counts.plot(kind='bar', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title("Cells per Donor", fontweight='bold', fontsize=12)
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=45)

# Clones per donor
clones_per_donor = cells_df.groupby('donor_id')['clone_id'].nunique().sort_index()
clones_per_donor.plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='black')
axes[1].set_title("Clones per Donor", fontweight='bold', fontsize=12)
axes[1].set_ylabel("Count")
axes[1].tick_params(axis='x', rotation=45)

# Heatmap: Stage x Donor
stage_donor = cells_df.groupby(['stage', 'donor_id']).size().unstack(fill_value=0)
sns.heatmap(stage_donor, ax=axes[2], cmap='YlOrRd', annot=True, fmt='d', cbar_kws={'label': 'Cells'})
axes[2].set_title("Cells (Stage x Donor)", fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig2_donor_structure.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELL 6: FIGURE - CELL TYPE DISTRIBUTION
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Cell Type Distribution")
print("="*80)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cell type counts
ct_counts = cells_df['cell_type'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(ct_counts)))
ct_counts.plot(kind='barh', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title("Cells per Cell Type", fontweight='bold', fontsize=12)
axes[0].set_xlabel("Count")

# Highlight influential cell types
influential = ground_truth.get('influential_celltypes', [])
for i, ct in enumerate(ct_counts.index):
    if ct in influential:
        axes[0].get_children()[i].set_edgecolor('red')
        axes[0].get_children()[i].set_linewidth(3)

axes[0].legend(['Influential (red border)'], loc='lower right')

# Cell type by stage heatmap
ct_stage = cells_df.groupby(['cell_type', 'stage']).size().unstack(fill_value=0)
sns.heatmap(ct_stage, ax=axes[1], cmap='Blues', annot=True, fmt='d', cbar_kws={'label': 'Cells'})
axes[1].set_title("Cell Types by Stage", fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig3_celltype_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nInfluential cell types (ground truth): {influential}")

---
## STEP 3: Visualize Ground Truth

In [ ]:
# ============================================================================
# CELL 7: FIGURE - GROUND TRUTH: STAGE CENTROIDS (Suite A)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Ground Truth - Stage Centroids (Suite A: Flow Field)")
print("="*80)

# Get stage centroids from ground truth
stage_centroids = ground_truth.get('stage_centroids', {})

if stage_centroids:
    # Extract first 2 dimensions for visualization
    centroids_2d = {stage: np.array(vec)[:2] for stage, vec in stage_centroids.items()}
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot centroids
    ax = axes[0]
    stages_list = list(centroids_2d.keys())
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(stages_list)))
    
    for i, (stage, pos) in enumerate(centroids_2d.items()):
        ax.scatter(pos[0], pos[1], s=400, c=[colors[i]], edgecolor='black', zorder=3, label=stage)
        ax.annotate(stage, (pos[0], pos[1]), fontsize=10, ha='center', va='bottom', 
                   xytext=(0, 10), textcoords='offset points')
    
    # Draw edges
    for src, tgt in stage_edges:
        if src in centroids_2d and tgt in centroids_2d:
            p1, p2 = centroids_2d[src], centroids_2d[tgt]
            ax.annotate('', xy=p2, xytext=p1,
                       arrowprops=dict(arrowstyle='->', color='gray', lw=2, alpha=0.7))
    
    ax.set_title("Stage Centroids in Latent Space (dim 0-1)", fontweight='bold', fontsize=12)
    ax.set_xlabel("Latent Dim 0")
    ax.set_ylabel("Latent Dim 1")
    ax.legend(loc='best')
    
    # Plot all cells colored by stage (sample)
    ax = axes[1]
    sample_cells = cells_df.sample(min(500, len(cells_df)), random_state=42)
    
    for i, stage in enumerate(stages_list):
        mask = sample_cells['stage'] == stage
        z_fused = np.stack(sample_cells.loc[mask, 'z_fused'].values)
        ax.scatter(z_fused[:, 0], z_fused[:, 1], s=20, c=[colors[i]], alpha=0.5, label=stage)
    
    ax.set_title("Cell Embeddings by Stage (sample)", fontweight='bold', fontsize=12)
    ax.set_xlabel("Latent Dim 0")
    ax.set_ylabel("Latent Dim 1")
    ax.legend(loc='best')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig4_stage_centroids.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No stage centroids in ground truth")

In [ ]:
# ============================================================================
# CELL 8: FIGURE - GROUND TRUTH: NICHE INFLUENCE (Suite B)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Ground Truth - Niche Influence Vectors (Suite B)")
print("="*80)

influence_vectors = ground_truth.get('influence_vectors', {})
influential_cts = ground_truth.get('influential_celltypes', [])

if influence_vectors:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Panel 1: Influence vector magnitudes
    ax = axes[0]
    magnitudes = {ct: np.linalg.norm(vec) for ct, vec in influence_vectors.items()}
    cts = list(magnitudes.keys())
    mags = list(magnitudes.values())
    bars = ax.barh(cts, mags, color='salmon', edgecolor='black')
    ax.set_title("Influence Vector Magnitudes", fontweight='bold', fontsize=12)
    ax.set_xlabel("Magnitude")
    
    # Panel 2: Influence vectors heatmap (first 8 dims)
    ax = axes[1]
    vec_matrix = np.array([influence_vectors[ct][:8] for ct in cts])
    sns.heatmap(vec_matrix, ax=ax, cmap='RdBu_r', center=0, 
                yticklabels=cts, xticklabels=[f'd{i}' for i in range(8)],
                cbar_kws={'label': 'Effect'})
    ax.set_title("Influence Vectors (dims 0-7)", fontweight='bold', fontsize=12)
    
    # Panel 3: Niche influence scores distribution
    ax = axes[2]
    ax.hist(cells_df['niche_influence_score'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    ax.axvline(cells_df['niche_influence_score'].mean(), color='red', linestyle='--', 
               label=f"Mean: {cells_df['niche_influence_score'].mean():.3f}")
    ax.set_title("Niche Influence Score Distribution", fontweight='bold', fontsize=12)
    ax.set_xlabel("Score")
    ax.set_ylabel("Count")
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig5_niche_influence_gt.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nInfluential cell types: {influential_cts}")
    print(f"Influence magnitudes: {magnitudes}")
else:
    print("No influence vectors in ground truth")

In [ ]:
# ============================================================================
# CELL 9: FIGURE - GROUND TRUTH: TRANSITIONS (Suite A)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Ground Truth - Transition Dynamics")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Transitions per edge
ax = axes[0]
edge_counts = transitions_df.groupby(['source_stage', 'target_stage']).size()
edge_labels = [f"{s}->{t}" for s, t in edge_counts.index]
ax.barh(edge_labels, edge_counts.values, color='lightgreen', edgecolor='black')
ax.set_title("Transitions per Stage Edge", fontweight='bold', fontsize=12)
ax.set_xlabel("Count")

# Panel 2: Transition vector magnitudes
ax = axes[1]
sample_trans = transitions_df.sample(min(200, len(transitions_df)), random_state=42)
z_src = np.stack(sample_trans['z_source'].values)
z_tgt = np.stack(sample_trans['z_target'].values)
trans_mags = np.linalg.norm(z_tgt - z_src, axis=1)
ax.hist(trans_mags, bins=30, color='orchid', edgecolor='black', alpha=0.7)
ax.axvline(trans_mags.mean(), color='red', linestyle='--', label=f"Mean: {trans_mags.mean():.3f}")
ax.set_title("Transition Vector Magnitudes", fontweight='bold', fontsize=12)
ax.set_xlabel("||z_target - z_source||")
ax.set_ylabel("Count")
ax.legend()

# Panel 3: Drift vs Diffusion visualization
ax = axes[2]
drift = ground_truth.get('drift_strength', 1.0)
diffusion = ground_truth.get('diffusion_strength', 0.2)
ax.bar(['Drift', 'Diffusion'], [drift, diffusion], color=['#2ecc71', '#e74c3c'], edgecolor='black')
ax.set_title(f"Dynamics Parameters ({DIFFICULTY})", fontweight='bold', fontsize=12)
ax.set_ylabel("Strength")
ax.text(0, drift + 0.05, f"{drift:.2f}", ha='center', fontsize=11)
ax.text(1, diffusion + 0.05, f"{diffusion:.2f}", ha='center', fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig6_transitions_gt.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDrift strength: {drift}")
print(f"Diffusion strength: {diffusion}")
print(f"Total transitions: {len(transitions_df)}")

In [ ]:
# ============================================================================
# CELL 10: FIGURE - SPATIAL STRUCTURE
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Spatial Structure")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sample cells for plotting
sample = cells_df.sample(min(500, len(cells_df)), random_state=42)

# Panel 1: Spatial positions colored by stage
ax = axes[0]
stages_list = sample['stage'].unique()
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(stages_list)))
for i, stage in enumerate(sorted(stages_list)):
    mask = sample['stage'] == stage
    ax.scatter(sample.loc[mask, 'x_spatial'], sample.loc[mask, 'y_spatial'], 
               s=15, c=[colors[i]], alpha=0.6, label=stage)
ax.set_title("Spatial Positions by Stage", fontweight='bold', fontsize=12)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.legend(loc='best', fontsize=8)

# Panel 2: Spatial positions colored by cell type
ax = axes[1]
ct_list = sample['cell_type'].unique()
ct_colors = plt.cm.Set3(np.linspace(0, 1, len(ct_list)))
for i, ct in enumerate(sorted(ct_list)):
    mask = sample['cell_type'] == ct
    ax.scatter(sample.loc[mask, 'x_spatial'], sample.loc[mask, 'y_spatial'], 
               s=15, c=[ct_colors[i]], alpha=0.6, label=ct)
ax.set_title("Spatial Positions by Cell Type", fontweight='bold', fontsize=12)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.legend(loc='best', fontsize=7, ncol=2)

# Panel 3: Niche influence score spatial
ax = axes[2]
sc = ax.scatter(sample['x_spatial'], sample['y_spatial'], 
                c=sample['niche_influence_score'], s=15, cmap='viridis', alpha=0.7)
plt.colorbar(sc, ax=ax, label='Niche Influence')
ax.set_title("Niche Influence Score (Spatial)", fontweight='bold', fontsize=12)
ax.set_xlabel("X")
ax.set_ylabel("Y")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig7_spatial_structure.png", dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 4: Data Splits and Loader

In [ ]:
# ============================================================================
# CELL 11: FIGURE - DATA SPLITS
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Cross-Validation Splits")
print("="*80)

folds = splits.get('folds', [])
n_folds = len(folds)

fig, axes = plt.subplots(1, n_folds, figsize=(5*n_folds, 4))
if n_folds == 1:
    axes = [axes]

all_donors = sorted(cells_df['donor_id'].unique())

for i, fold in enumerate(folds):
    ax = axes[i]
    train_d = set(fold.get('train_donors', []))
    val_d = set(fold.get('val_donors', []))
    test_d = set(fold.get('test_donors', []))
    
    colors = []
    for d in all_donors:
        if d in train_d:
            colors.append('#2ecc71')  # green
        elif d in val_d:
            colors.append('#f39c12')  # orange
        elif d in test_d:
            colors.append('#e74c3c')  # red
        else:
            colors.append('#bdc3c7')  # gray
    
    ax.barh(all_donors, [1]*len(all_donors), color=colors, edgecolor='black')
    ax.set_title(f"Fold {i+1}", fontweight='bold', fontsize=12)
    ax.set_xlabel("")
    ax.set_xlim(0, 1.5)
    ax.set_xticks([])
    
    # Add legend on first plot
    if i == 0:
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='#2ecc71', edgecolor='black', label='Train'),
            Patch(facecolor='#f39c12', edgecolor='black', label='Val'),
            Patch(facecolor='#e74c3c', edgecolor='black', label='Test'),
        ]
        ax.legend(handles=legend_elements, loc='upper right')

plt.suptitle("Donor-Held-Out Cross-Validation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig8_cv_splits.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal folds: {n_folds}")
for i, fold in enumerate(folds):
    print(f"  Fold {i+1}: train={len(fold.get('train_donors', []))}, "
          f"val={len(fold.get('val_donors', []))}, test={len(fold.get('test_donors', []))}")

---
## STEP 5: Model Training with Live Visualization

In [ ]:
# ============================================================================
# CELL 12: SETUP MODEL AND DATALOADERS
# ============================================================================
print("\n" + "="*80)
print("SETTING UP MODEL AND DATA")
print("="*80)

from stagebridge.data.loaders import get_dataloader
from stagebridge.pipelines.run_v1_full import StageBridgeV1Full

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {device}")

# Create model
model = StageBridgeV1Full(
    latent_dim=LATENT_DIM,
    niche_encoder_type="transformer",
    use_set_encoder=True,
    use_wes=True,
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

# Create dataloaders for fold 0
train_loader = get_dataloader(
    data_dir=str(DATA_DIR),
    fold=0,
    split="train",
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
)

val_loader = get_dataloader(
    data_dir=str(DATA_DIR),
    fold=0,
    split="val",
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
)

test_loader = get_dataloader(
    data_dir=str(DATA_DIR),
    fold=0,
    split="test",
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# ============================================================================
# CELL 13: TRAINING WITH LIVE VISUALIZATION
# ============================================================================
print("\n" + "="*80)
print("TRAINING WITH LIVE VISUALIZATION")
print("="*80)

import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm

# Training setup
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
criterion = nn.MSELoss()

# History
history = {
    'train_loss': [],
    'val_loss': [],
    'lr': [],
}

# Live figure
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plt.ion()  # Interactive mode

best_val_loss = float('inf')
best_model_state = None

for epoch in range(N_EPOCHS):
    # Training
    model.train()
    train_losses = []
    
    for batch in train_loader:
        z_src = batch['z_source'].to(device)
        z_tgt = batch['z_target'].to(device)
        niche = batch.get('niche_embedding', torch.zeros_like(z_src)).to(device)
        wes = batch.get('wes_features', torch.zeros(z_src.size(0), 1)).to(device)
        
        optimizer.zero_grad()
        z_pred = model(z_src, niche, wes)
        loss = criterion(z_pred, z_tgt)
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
    
    # Validation
    model.eval()
    val_losses = []
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for batch in val_loader:
            z_src = batch['z_source'].to(device)
            z_tgt = batch['z_target'].to(device)
            niche = batch.get('niche_embedding', torch.zeros_like(z_src)).to(device)
            wes = batch.get('wes_features', torch.zeros(z_src.size(0), 1)).to(device)
            
            z_pred = model(z_src, niche, wes)
            loss = criterion(z_pred, z_tgt)
            val_losses.append(loss.item())
            
            val_preds.append(z_pred.cpu().numpy())
            val_targets.append(z_tgt.cpu().numpy())
    
    # Record history
    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(scheduler.get_last_lr()[0])
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
    
    scheduler.step()
    
    # Update live plot
    clear_output(wait=True)
    
    for ax in axes.flat:
        ax.clear()
    
    # Panel 1: Loss curves
    ax = axes[0, 0]
    epochs_so_far = range(1, epoch + 2)
    ax.plot(epochs_so_far, history['train_loss'], 'b-', label='Train', linewidth=2)
    ax.plot(epochs_so_far, history['val_loss'], 'r-', label='Val', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.set_title(f'Training Progress (Epoch {epoch+1}/{N_EPOCHS})', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Panel 2: Learning rate
    ax = axes[0, 1]
    ax.plot(epochs_so_far, history['lr'], 'g-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Panel 3: Prediction vs Target (first 2 dims)
    ax = axes[0, 2]
    if val_preds:
        preds = np.concatenate(val_preds)
        targets = np.concatenate(val_targets)
        ax.scatter(targets[:, 0], preds[:, 0], alpha=0.5, s=10)
        ax.plot([-3, 3], [-3, 3], 'r--', linewidth=2)
        ax.set_xlabel('Target (dim 0)')
        ax.set_ylabel('Predicted (dim 0)')
        ax.set_title('Prediction vs Target', fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    # Panel 4: Residual distribution
    ax = axes[1, 0]
    if val_preds:
        residuals = preds - targets
        ax.hist(residuals.flatten(), bins=50, alpha=0.7, color='purple', edgecolor='black')
        ax.axvline(0, color='red', linestyle='--', linewidth=2)
        ax.set_xlabel('Residual')
        ax.set_ylabel('Count')
        ax.set_title(f'Residual Distribution (std={residuals.std():.4f})', fontweight='bold')
    
    # Panel 5: Per-dimension error
    ax = axes[1, 1]
    if val_preds:
        dim_errors = np.abs(residuals).mean(axis=0)
        ax.bar(range(len(dim_errors)), dim_errors, color='teal', edgecolor='black')
        ax.set_xlabel('Latent Dimension')
        ax.set_ylabel('MAE')
        ax.set_title('Error per Dimension', fontweight='bold')
    
    # Panel 6: Status text
    ax = axes[1, 2]
    ax.axis('off')
    status_text = f"""
    Epoch: {epoch+1}/{N_EPOCHS}
    
    Train Loss: {train_loss:.6f}
    Val Loss:   {val_loss:.6f}
    Best Val:   {best_val_loss:.6f}
    
    Learning Rate: {history['lr'][-1]:.6f}
    
    Difficulty: {DIFFICULTY.upper()}
    """
    ax.text(0.1, 0.9, status_text, transform=ax.transAxes, fontsize=12, 
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_title('Status', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"training_epoch_{epoch+1:02d}.png", dpi=100, bbox_inches='tight')
    display(fig)

plt.ioff()
plt.close()

# Save final training figure
print(f"\n Training complete!")
print(f"Best validation loss: {best_val_loss:.6f}")

In [ ]:
# ============================================================================
# CELL 14: FIGURE - FINAL TRAINING CURVES
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Final Training Curves")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, N_EPOCHS + 1)

# Loss curves
ax = axes[0]
ax.semilogy(epochs, history['train_loss'], 'b-', label='Train', linewidth=2, marker='o', markersize=4)
ax.semilogy(epochs, history['val_loss'], 'r-', label='Val', linewidth=2, marker='s', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Training & Validation Loss', fontweight='bold', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# Learning rate
ax = axes[1]
ax.plot(epochs, history['lr'], 'g-', linewidth=2, marker='^', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)

# Overfitting indicator
ax = axes[2]
gap = np.array(history['val_loss']) - np.array(history['train_loss'])
colors = ['green' if g < 0.01 else 'orange' if g < 0.05 else 'red' for g in gap]
ax.bar(epochs, gap, color=colors, edgecolor='black')
ax.axhline(0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Epoch')
ax.set_ylabel('Val - Train Loss')
ax.set_title('Generalization Gap', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig9_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal train loss: {history['train_loss'][-1]:.6f}")
print(f"Final val loss: {history['val_loss'][-1]:.6f}")
print(f"Best val loss: {best_val_loss:.6f}")

---
## STEP 6: Evaluation and Ground Truth Recovery

In [ ]:
# ============================================================================
# CELL 15: EVALUATION ON TEST SET
# ============================================================================
print("\n" + "="*80)
print("EVALUATION ON TEST SET")
print("="*80)

# Load best model
model.load_state_dict(best_model_state)
model.eval()

# Test evaluation
test_preds = []
test_targets = []
test_sources = []

with torch.no_grad():
    for batch in test_loader:
        z_src = batch['z_source'].to(device)
        z_tgt = batch['z_target'].to(device)
        niche = batch.get('niche_embedding', torch.zeros_like(z_src)).to(device)
        wes = batch.get('wes_features', torch.zeros(z_src.size(0), 1)).to(device)
        
        z_pred = model(z_src, niche, wes)
        
        test_preds.append(z_pred.cpu().numpy())
        test_targets.append(z_tgt.cpu().numpy())
        test_sources.append(z_src.cpu().numpy())

test_preds = np.concatenate(test_preds)
test_targets = np.concatenate(test_targets)
test_sources = np.concatenate(test_sources)

# Compute metrics
from scipy.stats import wasserstein_distance

mse = np.mean((test_preds - test_targets) ** 2)
mae = np.mean(np.abs(test_preds - test_targets))
w_dist = np.mean([wasserstein_distance(test_preds[:, i], test_targets[:, i]) 
                  for i in range(test_preds.shape[1])])

print(f"\nTest Metrics:")
print(f"  MSE: {mse:.6f}")
print(f"  MAE: {mae:.6f}")
print(f"  Wasserstein: {w_dist:.6f}")

In [ ]:
# ============================================================================
# CELL 16: FIGURE - TEST SET PREDICTIONS
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Test Set Predictions")
print("="*80)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Panel 1: Predicted vs Target (dim 0)
ax = axes[0, 0]
ax.scatter(test_targets[:, 0], test_preds[:, 0], alpha=0.5, s=15, c='steelblue')
ax.plot([-3, 3], [-3, 3], 'r--', linewidth=2, label='Perfect')
ax.set_xlabel('Target (dim 0)')
ax.set_ylabel('Predicted (dim 0)')
ax.set_title('Prediction vs Target (dim 0)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Predicted vs Target (dim 1)
ax = axes[0, 1]
ax.scatter(test_targets[:, 1], test_preds[:, 1], alpha=0.5, s=15, c='coral')
ax.plot([-3, 3], [-3, 3], 'r--', linewidth=2, label='Perfect')
ax.set_xlabel('Target (dim 1)')
ax.set_ylabel('Predicted (dim 1)')
ax.set_title('Prediction vs Target (dim 1)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Error distribution
ax = axes[0, 2]
errors = np.linalg.norm(test_preds - test_targets, axis=1)
ax.hist(errors, bins=30, color='purple', edgecolor='black', alpha=0.7)
ax.axvline(errors.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {errors.mean():.3f}')
ax.set_xlabel('Prediction Error (L2)')
ax.set_ylabel('Count')
ax.set_title('Error Distribution', fontweight='bold')
ax.legend()

# Panel 4: Transition vectors in 2D (sample)
ax = axes[1, 0]
n_show = min(100, len(test_sources))
for i in range(n_show):
    ax.arrow(test_sources[i, 0], test_sources[i, 1],
             test_preds[i, 0] - test_sources[i, 0],
             test_preds[i, 1] - test_sources[i, 1],
             head_width=0.05, head_length=0.02, fc='blue', ec='blue', alpha=0.3)
ax.set_xlabel('Dim 0')
ax.set_ylabel('Dim 1')
ax.set_title('Predicted Transitions (sample)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Panel 5: Per-dimension correlation
ax = axes[1, 1]
correlations = [np.corrcoef(test_preds[:, i], test_targets[:, i])[0, 1] 
                for i in range(min(LATENT_DIM, 16))]
ax.bar(range(len(correlations)), correlations, color='teal', edgecolor='black')
ax.axhline(0.8, color='red', linestyle='--', label='r=0.8')
ax.set_xlabel('Latent Dimension')
ax.set_ylabel('Correlation')
ax.set_title('Per-Dimension Correlation', fontweight='bold')
ax.set_ylim(0, 1)
ax.legend()

# Panel 6: Metrics summary
ax = axes[1, 2]
ax.axis('off')
metrics_text = f"""
TEST SET METRICS
{'='*30}

MSE:         {mse:.6f}
MAE:         {mae:.6f}
Wasserstein: {w_dist:.6f}

Mean Error:  {errors.mean():.6f}
Std Error:   {errors.std():.6f}

Mean Corr:   {np.mean(correlations):.4f}

Difficulty:  {DIFFICULTY.upper()}
"""
ax.text(0.1, 0.9, metrics_text, transform=ax.transAxes, fontsize=12, 
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig10_test_predictions.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELL 17: FIGURE - LATENT SPACE VISUALIZATION
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Latent Space Visualization (UMAP)")
print("="*80)

try:
    from umap import UMAP
    
    # Combine source, target, predicted
    all_z = np.vstack([test_sources, test_targets, test_preds])
    labels = ['Source'] * len(test_sources) + ['Target'] * len(test_targets) + ['Predicted'] * len(test_preds)
    
    # Fit UMAP
    print("Fitting UMAP...")
    umap = UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    z_2d = umap.fit_transform(all_z)
    
    n = len(test_sources)
    z_src_2d = z_2d[:n]
    z_tgt_2d = z_2d[n:2*n]
    z_pred_2d = z_2d[2*n:]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Panel 1: Source and Target
    ax = axes[0]
    ax.scatter(z_src_2d[:, 0], z_src_2d[:, 1], s=15, alpha=0.5, c='blue', label='Source')
    ax.scatter(z_tgt_2d[:, 0], z_tgt_2d[:, 1], s=15, alpha=0.5, c='green', label='Target')
    ax.set_title('Source vs Target States', fontweight='bold')
    ax.legend()
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    # Panel 2: Target and Predicted
    ax = axes[1]
    ax.scatter(z_tgt_2d[:, 0], z_tgt_2d[:, 1], s=15, alpha=0.5, c='green', label='Target')
    ax.scatter(z_pred_2d[:, 0], z_pred_2d[:, 1], s=15, alpha=0.5, c='red', label='Predicted')
    ax.set_title('Target vs Predicted States', fontweight='bold')
    ax.legend()
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    # Panel 3: All three
    ax = axes[2]
    ax.scatter(z_src_2d[:, 0], z_src_2d[:, 1], s=15, alpha=0.4, c='blue', label='Source')
    ax.scatter(z_tgt_2d[:, 0], z_tgt_2d[:, 1], s=15, alpha=0.4, c='green', label='Target')
    ax.scatter(z_pred_2d[:, 0], z_pred_2d[:, 1], s=15, alpha=0.4, c='red', label='Predicted')
    ax.set_title('All States (UMAP)', fontweight='bold')
    ax.legend()
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig11_umap_latent.png", dpi=150, bbox_inches='tight')
    plt.show()
    
except ImportError:
    print("UMAP not installed. Skipping latent visualization.")
    print("Install with: pip install umap-learn")

---
## STEP 7: Ground Truth Recovery Analysis

In [ ]:
# ============================================================================
# CELL 18: GROUND TRUTH RECOVERY METRICS
# ============================================================================
print("\n" + "="*80)
print("GROUND TRUTH RECOVERY ANALYSIS")
print("="*80)

recovery_metrics = {}

# Suite A: Flow field recovery
print("\nSuite A: Flow Field Dynamics")
print("-" * 40)

# Compute transition direction accuracy
pred_directions = test_preds - test_sources
true_directions = test_targets - test_sources

# Cosine similarity between predicted and true directions
dot_products = np.sum(pred_directions * true_directions, axis=1)
pred_norms = np.linalg.norm(pred_directions, axis=1) + 1e-8
true_norms = np.linalg.norm(true_directions, axis=1) + 1e-8
direction_cosines = dot_products / (pred_norms * true_norms)

direction_accuracy = (direction_cosines > 0.5).mean()
mean_cosine = direction_cosines.mean()

recovery_metrics['flow_direction_accuracy'] = direction_accuracy
recovery_metrics['flow_mean_cosine'] = mean_cosine

print(f"  Direction accuracy (cos > 0.5): {direction_accuracy:.4f}")
print(f"  Mean cosine similarity: {mean_cosine:.4f}")

# Suite B: Niche influence recovery (placeholder - needs attention extraction)
print("\nSuite B: Niche Influence Recovery")
print("-" * 40)
print("  (Requires attention weight extraction - see transformer analysis)")

# Suite C: Clone structure
print("\nSuite C: Clone Structure")
print("-" * 40)
print("  Clone divergence: {:.4f}".format(ground_truth.get('clone_divergence', 0.0)))

# Suite D: Spatial interactions
print("\nSuite D: Spatial Interactions")
print("-" * 40)
niche_strength = ground_truth.get('niche_influence_strength', 0.0)
print(f"  Niche influence strength: {niche_strength:.4f}")

print("\n" + "="*40)
print("RECOVERY SUMMARY")
print("="*40)
for k, v in recovery_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# ============================================================================
# CELL 19: FIGURE - FLOW FIELD RECOVERY
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Flow Field Recovery")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Direction cosine distribution
ax = axes[0]
ax.hist(direction_cosines, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(0, color='gray', linestyle='--', linewidth=2)
ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold (0.5)')
ax.axvline(mean_cosine, color='green', linestyle='-', linewidth=2, label=f'Mean: {mean_cosine:.3f}')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Count')
ax.set_title('Direction Accuracy Distribution', fontweight='bold')
ax.legend()

# Panel 2: Magnitude comparison
ax = axes[1]
ax.scatter(true_norms, pred_norms, alpha=0.5, s=15, c='coral')
max_norm = max(true_norms.max(), pred_norms.max())
ax.plot([0, max_norm], [0, max_norm], 'k--', linewidth=2, label='Perfect')
ax.set_xlabel('True Transition Magnitude')
ax.set_ylabel('Predicted Transition Magnitude')
ax.set_title('Transition Magnitude Recovery', fontweight='bold')
ax.legend()

# Panel 3: Sample flow vectors
ax = axes[2]
n_show = min(50, len(test_sources))
for i in range(n_show):
    # True direction (green)
    ax.arrow(test_sources[i, 0], test_sources[i, 1],
             true_directions[i, 0] * 0.8, true_directions[i, 1] * 0.8,
             head_width=0.03, head_length=0.01, fc='green', ec='green', alpha=0.5)
    # Predicted direction (red)
    ax.arrow(test_sources[i, 0], test_sources[i, 1],
             pred_directions[i, 0] * 0.8, pred_directions[i, 1] * 0.8,
             head_width=0.03, head_length=0.01, fc='red', ec='red', alpha=0.5)

ax.set_xlabel('Dim 0')
ax.set_ylabel('Dim 1')
ax.set_title('True (green) vs Predicted (red) Flow', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig12_flow_recovery.png", dpi=150, bbox_inches='tight')
plt.show()

---
## FINAL SUMMARY

In [ ]:
# ============================================================================
# CELL 20: FINAL SUMMARY AND FIGURE GALLERY
# ============================================================================
print("\n" + "="*80)
print("PIPELINE COMPLETE - FINAL SUMMARY")
print("="*80)

print(f"\nConfiguration:")
print(f"  Difficulty: {DIFFICULTY}")
print(f"  Cells: {N_CELLS}")
print(f"  Donors: {N_DONORS}")
print(f"  Epochs: {N_EPOCHS}")

print(f"\nFinal Metrics:")
print(f"  MSE: {mse:.6f}")
print(f"  MAE: {mae:.6f}")
print(f"  Wasserstein: {w_dist:.6f}")
print(f"  Flow Direction Accuracy: {direction_accuracy:.4f}")

# List all generated figures
print(f"\nGenerated Figures:")
for fig_path in sorted(OUTPUT_DIR.glob("fig*.png")):
    print(f"  {fig_path.name}")

print(f"\nOutput directory: {OUTPUT_DIR}")
print("\n" + "="*80)
print("DONE!")
print("="*80)

In [ ]:
# ============================================================================
# CELL 21: FIGURE GALLERY (display all)
# ============================================================================
print("\n" + "="*80)
print("FIGURE GALLERY")
print("="*80)

from IPython.display import Image

for fig_path in sorted(OUTPUT_DIR.glob("fig*.png")):
    print(f"\n{fig_path.stem}")
    print("-" * 60)
    display(Image(filename=str(fig_path), width=800))